# 02 - Dataset generation

The 2800 forward solves that everything after this notebook consumes: 2000 train,
400 val, 400 test, plus the 32 cached incident solves the whole pipeline shares.
This is §11.2 step 6.

**This is the expensive notebook.** It is also the one whose output is hardest to check
after the fact, because a dataset that is subtly wrong -- a swapped axis, an incident
field subtracted from the wrong source, a phasor written at the wrong index -- still
trains, still converges, and produces a surrogate for an operator nobody asked for. So
the last third of the notebook re-runs the solver on stored samples and compares.

**Order of operations.** Project the size and the wall clock *before* generating, check
the disk has room, run the incident solves once, then the three splits. Every step is
skip-if-exists, so an interrupted run resumes at split granularity.

**Platform.** On Modal this is the job to run headless rather than in a notebook -- a
several-hour cell in a browser tab is a way to lose several hours:

```
modal run modal_app.py::generate_all
```

The data lands in the Modal Volume at the same path `bootstrap.setup()` reports below,
so a notebook session started afterwards sees the files with nothing to copy.

In [ ]:
# The repo root holds bootstrap.py; these notebooks live one level down in notebooks/.
# bootstrap.setup() puts the repo on sys.path, installs anything missing, picks the
# device, finds a persistent data directory, and turns TF32 off.  It is the only
# platform-aware code in this notebook.
import pathlib
import sys

_here = pathlib.Path.cwd()
_root = next((p for p in (_here, *_here.parents) if (p / "bootstrap.py").exists()), None)
assert _root is not None, "run this notebook from inside the fno-wave-inverse checkout"
if str(_root) not in sys.path:
    sys.path.insert(0, str(_root))

import bootstrap

E = bootstrap.setup()
DEV = E.device

In [ ]:
import json
import math
import time

import matplotlib.pyplot as plt
import numpy as np
import torch
from tqdm.auto import tqdm

%matplotlib inline
plt.rcParams.update({"figure.dpi": 110, "savefig.dpi": 170, "font.size": 9,
                     "figure.facecolor": "white", "axes.grid": True,
                     "grid.alpha": 0.25, "axes.axisbelow": True})

from src import config as cfg


def _np(v):
    # Anything printable/plottable, as a numpy array.  The library returns torch
    # tensors, numpy arrays and lists interchangeably depending on the entry point.
    if torch.is_tensor(v):
        return v.detach().cpu().numpy()
    return np.asarray(v)


def savefig(fig, name):
    p = E.figures / name
    fig.savefig(p, bbox_inches="tight")
    print("wrote", p)
    return p


def dump(obj, name):
    p = E.results / name
    p.write_text(json.dumps(obj, indent=1, default=float))
    print("wrote", p)
    return p


def table(rows, headers):
    w = [max(len(str(h)), *(len(f"{r[i]}") for r in rows)) if rows else len(str(h))
         for i, h in enumerate(headers)]
    line = "  ".join(f"{h:>{w[i]}}" for i, h in enumerate(headers))
    print(line)
    print("-" * len(line))
    for r in rows:
        print("  ".join(f"{r[i]:>{w[i]}}" for i in range(len(headers))))

In [ ]:
# The three splits and the checkpoint the later notebooks read.  Nothing here writes.
paths = {k: E.datasets / f"{k}.h5" for k in ("train", "val", "test")}
for k, p in paths.items():
    print(f"{k:>6}  {'present' if p.exists() else 'MISSING':>7}  "
          f"{(p.stat().st_size / 1e9 if p.exists() else 0):6.2f} GB  {p}")

CKPT_DIR = E.checkpoints / "full"
CKPT = CKPT_DIR / "best.pt"
print(f"\nckpt   {'present' if CKPT.exists() else 'MISSING':>7}  {CKPT}")

## What it will cost, before spending it

`projected_size` is arithmetic on the config, not a measurement: 20 frequencies x 2
components of `128^2` complex64 is 2.6 MB of phasors per sample, and 32 receivers x 2
components x 1408 steps of float32 is 0.36 MB of A-scans. `calibrate` measures this GPU's
actual cell-step throughput on a 50-step run, which turns the cell-step count into hours.

The A-scans are 12% of the volume and are *not* redundant with the phasors: the phasors
are on the network grid at 20 frequencies, and the inversion's data is the receiver time
series, from which the phasors are derived by a deconvolution that cannot be run backwards
to recover a trace. If the projection comes out too large, `M_FREQ` is the lever, and the
top of the band is the least informative part of it.

In [ ]:
from src.data import generate as G

N = {"train": cfg.N_TRAIN, "val": cfg.N_VAL, "test": cfg.N_TEST}
N_TOTAL = sum(N.values())
print(f"splits: {N}   total {N_TOTAL} samples\n")
G.print_projection(N_TOTAL)

In [ ]:
import shutil

thr = None
if DEV.startswith("cuda"):
    thr = G.calibrate(device=DEV)
    print(f"measured throughput {thr/1e9:.2f} G cell-steps/s on {E.gpu_name}\n")
    G.print_projection(N_TOTAL, throughput_cell_steps_per_s=thr)
else:
    print("CPU: skipping calibration.  Generating 2800 samples on CPU is not "
          "practical -- this notebook wants a GPU.\n")

need = G.projected_size(N_TOTAL)["total"]
du = shutil.disk_usage(E.data)
print(f"\ndisk at {E.data}")
print(f"  free  {du.free/1e9:8.2f} GB")
print(f"  need  {need/1e9:8.2f} GB")
print(f"  {'OK' if du.free > 1.25 * need else 'NOT ENOUGH ROOM (want 25% headroom)'}")
if not E.persistent:
    print("\nWARNING: bootstrap reports this data directory is NOT persistent.\n"
          "Generating here means losing it when the container stops.  Mount a\n"
          "Volume first -- see modal_app.py.")

## The incident field, once

Eight source positions x four Poisson ratios = 32 solves of the *homogeneous* domain.
They are run once, cached in every split file, and shared by every sample that uses that
acquisition -- 2000 training samples draw on 24 of these 32 solves, so caching them is a
factor of ~60 on the incident cost, and more importantly it makes the scattered field
`u_tot - u_inc` an exact difference of two solves on the same grid with the same time
stepping. Recomputing the incident field per sample would leave a residual at the level of
the solver's own reproducibility, which is not zero once the batch composition changes.

Two normalisation scales come out of this, and the distinction matters:

- `scale` is `max|u_inc|` over the whole domain, dominated by the near-source
  singularity. This is what the network's inputs and targets are divided by.
- `scale_recv` is `max|u_inc|` over the 32-receiver ring, three cells inside the absorber,
  where the field has already spread. This is what the receiver misfit uses.

They differ by about two orders of magnitude. Using the domain scale for the receiver
misfit would divide the residual by a number 100x too large and make the measurement term
silently negligible -- the loss would still go down, and the ring would be unconstrained.

In [ ]:
INC_CACHE = E.datasets / "incident.npz"

if INC_CACHE.exists():
    z = np.load(INC_CACHE)
    inc = {k: z[k] for k in ("phasors", "ascans", "scale", "scale_recv")}
    print(f"loaded cached incident solves from {INC_CACHE}")
else:
    t0 = time.perf_counter()
    inc = G.run_incident(device=DEV, progress=tqdm)
    print(f"32 incident solves in {(time.perf_counter()-t0)/60:.1f} min")
    np.savez(INC_CACHE, **inc)
    print("cached to", INC_CACHE)

for k, v in inc.items():
    print(f"  {k:>11}  {str(v.shape):>28}  {v.dtype}")

In [ ]:
sc, scr = inc["scale"], inc["scale_recv"]
fr = np.asarray(cfg.FREQS)

fig, ax = plt.subplots(1, 2, figsize=(9.4, 3.1))
for s in range(cfg.N_SRC):
    ax[0].semilogy(fr, sc[s, 0], lw=0.8, alpha=0.8)
    ax[0].semilogy(fr, scr[s, 0], lw=0.8, alpha=0.8, ls="--")
ax[0].semilogy([], [], c="0.3", lw=0.8, label="domain max (solid)")
ax[0].semilogy([], [], c="0.3", lw=0.8, ls="--", label="receiver-ring max (dashed)")
ax[0].set(xlabel="f / f_c", ylabel="max |u_inc|",
          title=f"incident amplitude, nu = {cfg.NU_LIST[0]}, all 8 sources")
ax[0].legend(fontsize=7.5)

ratio = sc / np.maximum(scr, 1e-30)
ax[1].semilogy(fr, ratio.reshape(-1, len(fr)).T, lw=0.7, alpha=0.6)
ax[1].set(xlabel="f / f_c", ylabel="domain max / ring max",
          title=f"ratio of the two scales\nmedian {np.median(ratio):.0f}x over all "
                f"{ratio.shape[0]*ratio.shape[1]} (source, nu)")
fig.tight_layout()
savefig(fig, "02_incident_scales.png")
plt.show()

print(f"ratio: min {ratio.min():.1f}  median {np.median(ratio):.1f}  "
      f"max {ratio.max():.1f}")

## Generating the three splits

The split boundary that matters is the source pool. `SRC_TRAIN` excludes
`SRC_HELDOUT = (3, 6)`; train and val draw from the remaining six sources and the test
split draws from all eight. So the test split contains illuminations the network has never
seen, and notebook 05 can report the inversion success rate split by seen/held-out source
as a generalisation number rather than an interpolation number.

Each split reuses the `incident` dict computed above, so all three files carry byte-identical
incident caches and a case loaded from one split can be compared against a model trained on
another without a units mismatch.

In [ ]:
print(f"train/val sources: {cfg.SRC_TRAIN}")
print(f"held out from train/val: {cfg.SRC_HELDOUT}")
print(f"test sources: {tuple(range(cfg.N_SRC))}\n")

for split, n in N.items():
    p = paths[split]
    if p.exists():
        with __import__("h5py").File(p, "r") as f:
            print(f"{split:>6}  exists, {int(f.attrs['n_samples'])} samples, "
                  f"src_pool {list(f.attrs['src_pool'])}, "
                  f"{float(f.attrs['generation_seconds'])/60:.1f} min  -- skipping")
        continue
    print(f"\n=== {split}: {n} samples -> {p}")
    G.generate(str(p), n, split=split, device=DEV, incident=inc, progress=tqdm)

In [ ]:
import h5py

# assert_compatible refuses to read a dataset generated under a different config.  It
# compares every key in generate._SNAPSHOT_KEYS, so a change to the grid, the band, the
# time step or the interface width invalidates the file rather than silently shifting
# what the labels mean.
rows = []
for split, p in paths.items():
    if not p.exists():
        rows.append((split, "MISSING", "-", "-", "-", "-"))
        continue
    with h5py.File(p, "r") as f:
        G.assert_compatible(f)
        th = f["samples/theta"][:]
        si = f["samples/src_idx"][:]
        rows.append((split, f"{int(f.attrs['n_samples'])}",
                     f"{p.stat().st_size/1e9:.2f} GB",
                     f"{th[:, 2].min():.3f}-{th[:, 2].max():.3f}",
                     f"{sorted(set(int(v) for v in si))}",
                     f"{float(f.attrs['generation_seconds'])/60:.1f} min"))
table(rows, ["split", "n", "size", "R range", "sources used", "wall"])

with h5py.File(paths["train"], "r") as f:
    print("\nfile layout")
    f.visit(lambda k: print("  ", k))

## §11.2 step 6: re-run the solver and compare

The check that catches everything the layout inspection above cannot. Twenty stored
samples are re-solved from their stored `theta`, `src_idx` and `nu_idx` alone, the *stored*
incident phasors are subtracted, and the result is compared to the stored scattered
phasors.

The gate is `rel-L2 < 1e-5`, which is far tighter than any physics tolerance and is
deliberately so. The solver is deterministic: given the same material fields, the same
source and the same time stepping it produces the same floats, *independent of batch
composition* -- all stepping ops are per-sample elementwise, and the harmonic-average
guard is the absolute constant `cfg.MU_HARMONIC_FLOOR = 1e-12` for exactly this reason.
Anything above round-off means the file does not describe the run that produced it.

If this fails, the likely causes in order: the incident cache indexed with `(nu_idx,
src_idx)` instead of `(src_idx, nu_idx)`; `nu_idx` read as a Poisson ratio rather than an
index into `NU_LIST`; the component and frequency axes transposed on write; or a dataset
generated before the batch-dependent floor fix (see below).

Fixed 2026-09-20: `ElasticFDTD2D` used `VOID_STIFFNESS_FLOOR * mu.max()` over the batch
as the harmonic-average floor, so the same sample solved differently with different batch
mates (median 0 / max 0.366 bimodal: same-floor batches reproduced bitwise, mixed-nu
batches did not). The floor is now absolute and inert, snapshotted as `MU_HARMONIC_FLOOR`,
and `tests/test_solver_batch.py` pins batch-independence. Datasets generated before this
fix fail this check by design -- regenerate them; `assert_compatible` refuses old files.

In [ ]:
from src.solver.fdtd_elastic import ElasticFDTD2D
from src.solver import harmonic as H

N_SPOT = 20
SPOT_BATCH = 4

with h5py.File(paths["test"] if paths["test"].exists() else paths["train"], "r") as f:
    n = int(f.attrs["n_samples"])
    pick = np.linspace(0, n - 1, N_SPOT).astype(int)
    th_all = f["samples/theta"][:][pick]
    s_all = f["samples/src_idx"][:][pick].astype(int)
    j_all = f["samples/nu_idx"][:][pick].astype(int)
    inc_ph = f["incident/phasors"][:]
    stored = np.stack([f["samples/us_phasors"][int(i)] for i in pick])

om = H.omegas_tensor(DEV)
errs = []
for lo in tqdm(range(0, N_SPOT, SPOT_BATCH), desc="re-solving"):
    hi = min(N_SPOT, lo + SPOT_BATCH)
    th = torch.as_tensor(th_all[lo:hi], device=DEV)
    nu_vals = torch.tensor([cfg.NU_LIST[j] for j in j_all[lo:hi]], device=DEV)
    chi = G.fine_chi(th, device=DEV)
    lam, mu, rho = G._materials(chi, nu_vals)
    sim = ElasticFDTD2D(lam, mu, rho)
    src = [cfg.net_to_fine(*cfg.SOURCES_NET[s]) for s in s_all[lo:hi]]
    res = sim.run(src, nt=cfg.NT, recv_yx=cfg.RECEIVERS_NET, omegas=om)
    u_tot = H.displacement_from_field(res.phasors, omegas=om)
    u_inc = torch.as_tensor(
        np.stack([inc_ph[s, j] for s, j in zip(s_all[lo:hi], j_all[lo:hi])]),
        device=DEV)
    fresh = (u_tot - u_inc).cpu().numpy()
    ref = stored[lo:hi]
    num = np.linalg.norm((fresh - ref).reshape(hi - lo, -1), axis=1)
    den = np.linalg.norm(ref.reshape(hi - lo, -1), axis=1)
    errs.extend((num / np.maximum(den, 1e-30)).tolist())

errs = np.asarray(errs)
GATE_SPOT = 1e-5
print(f"\nrel-L2 over {N_SPOT} re-solved samples")
print(f"  median {np.median(errs):.3e}   max {errs.max():.3e}   gate {GATE_SPOT:.0e}")
print(f"  {'PASS' if errs.max() < GATE_SPOT else 'FAIL'}  "
      f"the file describes the run that produced it")

In [ ]:
i_worst = int(errs.argmax())
u_ref = stored[i_worst]                              # [2, M, ny, nx] complex
m_show = [0, cfg.M_FREQ // 2, cfg.M_FREQ - 1]

fig, ax = plt.subplots(2, 4, figsize=(11.5, 5.4))
for c, m in enumerate(m_show):
    a = np.abs(u_ref[0, m])
    ax[0, c].imshow(a, origin="lower", cmap="magma")
    ax[0, c].set_title(f"|u_s,x|  f = {cfg.FREQS[m]:.2f} f_c", fontsize=8)
    ax[1, c].imshow(np.angle(u_ref[0, m]), origin="lower", cmap="twilight",
                    vmin=-np.pi, vmax=np.pi)
    ax[1, c].set_title("arg u_s,x", fontsize=8)
for a_ in ax.ravel():
    a_.set_xticks([]); a_.set_yticks([]); a_.grid(False)

ax[0, 3].semilogy(range(len(errs)), np.sort(errs)[::-1], "o-", ms=3)
ax[0, 3].axhline(GATE_SPOT, ls="--", c="C3", lw=1.0, label="gate 1e-5")
ax[0, 3].set(xlabel="sample (sorted)", ylabel="rel-L2 vs stored",
             title="re-solve agreement")
ax[0, 3].legend(fontsize=7.5)
ax[0, 3].grid(True, alpha=0.25)

amp = np.linalg.norm(stored.reshape(N_SPOT, -1), axis=1)
ax[1, 3].semilogy(th_all[:, 2], amp, "o", ms=4)
ax[1, 3].set(xlabel="R (physical)", ylabel="||u_s||_2",
             title="scattered amplitude carries R\n(this is why the target is not "
                   "self-normalised)")
ax[1, 3].grid(True, alpha=0.25)

fig.suptitle(f"stored sample {int(np.linspace(0, 1, N_SPOT)[i_worst]*100):d}th "
             f"percentile of the re-solve error", fontsize=9)
fig.tight_layout()
savefig(fig, "02_spot_check.png")
plt.show()

## The scattered field really is a difference, not a difference plus a constant

An independent check on the incident subtraction, and one that does not depend on any
stored array. Shrink the void radius towards zero at a fixed centre and watch the scattered
amplitude. In the Rayleigh regime the scattered amplitude goes as `R^2`, so a log-log fit
should give a slope near 2, and -- the point of the test -- it should keep falling. A
constant offset in the subtraction (wrong source, wrong nu, an off-by-one in the cache
index) is invisible at `R = 0.8 lambda_s` where the scattered field is large, and shows up
as a **floor** here.

The smallest radii in the sweep are smaller than the interface transition itself.
`EPS_INTERFACE_FINE_CELLS = 0.375` puts the 10-90% width at 1.65 fine cells, and
`R = 0.025 lambda_s` is 0.4 fine cells across, so down there `chi` never reaches 1 and the
"void" is a soft dimple. (At production radii it does reach 1 -- that is the point of the
narrow interface, and check 7 is what it bought.) The dimple is not a defect of the test:
the field still has to go to zero smoothly, and it is the floor that is being looked for,
not the exponent at the last point.

In [ ]:
radii_ls = np.array([0.40, 0.30, 0.20, 0.14, 0.10, 0.05, 0.025])
S_IDX, J_IDX = 0, 0
lam_s = cfg.cs_over_cp(cfg.NU_LIST[J_IDX]) / cfg.FC
xc, yc = 0.55 * cfg.L_DOMAIN, 0.45 * cfg.L_DOMAIN

th = torch.tensor([[xc, yc, float(r) * lam_s] for r in radii_ls],
                  dtype=torch.float32, device=DEV)
nu_vals = torch.full((len(radii_ls),), cfg.NU_LIST[J_IDX], device=DEV)
chi = G.fine_chi(th, device=DEV)
lam, mu, rho = G._materials(chi, nu_vals)
sim = ElasticFDTD2D(lam, mu, rho)
res = sim.run([cfg.net_to_fine(*cfg.SOURCES_NET[S_IDX])] * len(radii_ls),
              nt=cfg.NT, recv_yx=cfg.RECEIVERS_NET, omegas=om)
u_tot = H.displacement_from_field(res.phasors, omegas=om)
u_inc1 = torch.as_tensor(inc["phasors"][S_IDX, J_IDX], device=DEV).unsqueeze(0)
amp = (u_tot - u_inc1).abs().pow(2).sum(dim=(1, 2, 3, 4)).sqrt().cpu().numpy()

k = np.polyfit(np.log(radii_ls), np.log(amp), 1)
slope = float(k[0])
print(f"log-log slope of ||u_s|| vs R : {slope:.3f}   (Rayleigh amplitude ~ R^2)")
print(f"amplitude falls by {amp[0]/amp[-1]:.1f}x over a {radii_ls[0]/radii_ls[-1]:.0f}x "
      f"radius range -- no floor")

fig, ax = plt.subplots(figsize=(5.6, 3.2))
ax.loglog(radii_ls, amp, "o-", ms=5, label="||u_s||_2")
ax.loglog(radii_ls, amp[0] * (radii_ls / radii_ls[0]) ** 2.0, ls="--", c="0.45",
          lw=1.0, label="slope 2")
ax.axvline(cfg.EPS_TRANSITION_FACTOR * cfg.EPS_INTERFACE_PHYS / lam_s, ls=":",
           c="C3", lw=1.0, label="10-90% interface width")
ax.axvline(cfg.R_MIN_LS, ls="-.", c="C2", lw=1.0, label=f"R_MIN_LS = {cfg.R_MIN_LS}")
ax.set(xlabel="R / lambda_s", ylabel="scattered amplitude",
       title=f"incident subtraction has no offset\nfitted slope {slope:.2f}")
ax.legend(fontsize=7.5)
fig.tight_layout()
savefig(fig, "02_incident_subtraction.png")
plt.show()

## Wrap-around: is the DFT window long enough?

The third of the three named failure modes of the time-harmonic reduction. The phasors come
from a DFT run over a finite window of `NT = 1408` steps; anything still ringing at the end
of that window aliases back onto the band. `tail_energy_fraction` measures the energy in the
last 10% of each A-scan as a fraction of the total. Check 3 of notebook 01 established that
the absorber works on a clean domain; this measures the same thing on the real dataset,
where a void near the ring can hold energy longer than a homogeneous domain does.

In [ ]:
with h5py.File(paths["train"], "r") as f:
    a = torch.from_numpy(f["samples/ascans"][:64])
    th64 = f["samples/theta"][:64]
frac = _np(H.tail_energy_fraction(a))
frac = frac.reshape(frac.shape[0], -1).max(axis=1) if frac.ndim > 1 else frac

print(f"tail energy fraction over {len(frac)} samples")
print(f"  median {np.median(frac):.3e}   p90 {np.quantile(frac, 0.9):.3e}   "
      f"max {frac.max():.3e}")
print(f"  {'OK' if frac.max() < 1e-3 else 'HIGH -- lengthen T_END or thicken the absorber'}")

fig, ax = plt.subplots(1, 2, figsize=(9.2, 3.0))
ax[0].hist(np.log10(np.maximum(frac, 1e-12)), bins=24, color="C0", alpha=0.85)
ax[0].set(xlabel="log10 tail energy fraction", ylabel="samples",
          title="wrap-around margin, 64 training samples")
tr = _np(a[int(frac.argmax())])
t_ax = np.arange(tr.shape[-1]) * cfg.DT
for r in range(0, tr.shape[0], 8):
    ax[1].plot(t_ax, tr[r, 0] / max(abs(tr[r, 0]).max(), 1e-30) + 1.1 * (r // 8),
               lw=0.6)
ax[1].axvspan(0.9 * t_ax[-1], t_ax[-1], color="C3", alpha=0.10, label="tail window")
ax[1].set(xlabel="t / T_p", yticks=[], ylabel="receiver (every 8th)",
          title=f"worst sample, R = {th64[int(frac.argmax()), 2]:.3f}")
ax[1].legend(fontsize=8)
fig.tight_layout()
savefig(fig, "02_wraparound.png")
plt.show()

## The loader, end to end

Last: the path the training loop will actually take. Three things are being confirmed, all
of which have silent failure modes.

1. **Shapes and dtypes** through `WaveDataset` -> `make_loader` -> `batch_to_model`. The
   input has 12 channels, the target 4, and the frequency axis is folded into the batch, so
   `B = batch_size * n_freq`.
2. **Frequency subsetting is resampled per epoch.** `set_epoch` reseeds it, and that only
   works because `make_loader` sets `persistent_workers=False`. With persistent workers the
   dataset object is copied into each worker once, mutating `self.epoch` in the parent is
   invisible, and every epoch trains on the same 4 frequencies per sample. The symptom
   would be a validation curve that plateaus early for no visible reason -- so the two
   epochs below must give different subsets.
3. **Input and target share one divisor.** Both are divided by `incident/scale`, never by
   their own norm. Dividing the target by its own norm is the one normalisation that must
   not happen: the scattered amplitude carries the radius, and normalising it away leaves
   the inversion with nothing to recover `R` from.

In [ ]:
from src.data.dataset import WaveDataset, batch_to_model, make_loader

ds = WaveDataset(str(paths["train"]), train=True)
print(f"{len(ds)} samples, {ds.n_freq} of {cfg.M_FREQ} frequencies per sample per epoch")

ds.set_epoch(0)
s0 = ds[0]
ds.set_epoch(1)
s1 = ds[0]
print(f"\nsample 0 frequencies, epoch 0: {[round(float(v), 3) for v in s0['freqs']]}")
print(f"sample 0 frequencies, epoch 1: {[round(float(v), 3) for v in s1['freqs']]}")
same = torch.equal(s0["freqs"], s1["freqs"])
print(f"  {'FAIL -- set_epoch is not reseeding' if same else 'OK'}  subsets differ")

print("\nper-sample tensors")
for k, v in s0.items():
    print(f"  {k:>10}  {str(tuple(v.shape)):>22}  {str(v.dtype):>16}")

In [ ]:
ld = make_loader(ds, batch_size=4, num_workers=0)
batch = next(iter(ld))
x, y = batch_to_model(batch)
print(f"x {tuple(x.shape)} {x.dtype}      ({cfg.C_IN} channels expected)")
print(f"y {tuple(y.shape)} {y.dtype}      ({cfg.C_OUT} channels expected)")
print(f"rows = batch {4} x n_freq {ds.n_freq} = {4 * ds.n_freq}")
assert x.shape[1] == cfg.C_IN and y.shape[1] == cfg.C_OUT
assert x.shape[0] == y.shape[0] == 4 * ds.n_freq

from src import features as feat

names = feat.INPUT_CHANNELS
print("\nper-channel statistics of one batch")
table([(names[c], f"{float(x[:, c].mean()):+.4f}", f"{float(x[:, c].std()):.4f}",
        f"{float(x[:, c].min()):+.3f}", f"{float(x[:, c].max()):+.3f}")
       for c in range(x.shape[1])],
      ["channel", "mean", "std", "min", "max"])

# the target's amplitude spread across the batch is the radius information
rows_norm = y.flatten(1).norm(dim=1)
print(f"\ntarget row norms: min {float(rows_norm.min()):.4f}  "
      f"max {float(rows_norm.max()):.4f}  ratio {float(rows_norm.max()/rows_norm.min()):.1f}x")
print("A self-normalised target would make every one of these 1.0 and delete the "
      "radius.")
ds.close()

In [ ]:
record = {
    "device": DEV, "gpu": E.gpu_name, "data_dir": str(E.data),
    "splits": {k: dict(n=N[k], path=str(paths[k]), exists=paths[k].exists(),
                       bytes=(paths[k].stat().st_size if paths[k].exists() else 0))
               for k in N},
    "projected_bytes": G.projected_size(N_TOTAL),
    "throughput_cell_steps_per_s": thr,
    "spot_check": dict(n=N_SPOT, median=float(np.median(errs)),
                       max=float(errs.max()), gate=GATE_SPOT,
                       passed=bool(errs.max() < GATE_SPOT)),
    "incident_subtraction_slope": slope,
    "tail_energy_fraction": dict(median=float(np.median(frac)), max=float(frac.max())),
    "scale_ratio_domain_over_ring": float(np.median(ratio)),
}
dump(record, "02_dataset_generation.json")
print("\nDataset ready.  Notebook 03 trains on it.")